In [37]:
import torch
import pandas as pd
from torch_geometric.data import Data, HeteroData
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import RandomLinkSplit
import torch.nn as nn
from torch.utils.data import DataLoader
from torch_geometric.loader import LinkNeighborLoader
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from tqdm import tqdm

import utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [48]:
def df_to_homo_two_rel(df: pd.DataFrame):
    # 1) глобальные id для узлов: различаем по (entity_type, raw_id)
    #    (если ты хочешь 1 тип узлов, но НЕ сливать разные типы с одинаковым числом)
    nodes = pd.concat([
        df[["type_entity_1","id_entity_1"]].rename(columns={"type_entity_1":"t","id_entity_1":"id"}),
        df[["type_entity_2","id_entity_2"]].rename(columns={"type_entity_2":"t","id_entity_2":"id"}),
    ], axis=0).drop_duplicates()

    key = list(zip(nodes["t"].astype(str), nodes["id"].astype(int)))
    node_map = {k:i for i,k in enumerate(key)}
    num_nodes = len(node_map)

    # 2) edge_index
    src = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_1"], df["id_entity_1"])]
    dst = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_2"], df["id_entity_2"])]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # 3) edge_type (2 предиката -> 0/1)
    preds = df["predicate"].astype(str).unique().tolist()
    pred2id = {p:i for i,p in enumerate(sorted(preds))}
    edge_type = torch.tensor([pred2id[p] for p in df["predicate"].astype(str)], dtype=torch.long)

    edge_index, edge_type = to_undirected(edge_index, edge_type)
    edge_type[edge_type == 2] = 1 #подумать потом

    num_nodes = len(node_map)
    data = Data(edge_index=edge_index, num_nodes=num_nodes, edge_type=edge_type)
    return data, node_map, pred2id

class DistMult(nn.Module):
    def __init__(self, num_nodes, num_rels, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hidden_dim)
        self.rel = nn.Embedding(num_rels, hidden_dim)

    def forward(self, item):
        return self.emb(torch.tensor(item))

    def decode(self, edge_label_index, edge_type):

        src = self.emb(edge_label_index[0])
        dst = self.emb(edge_label_index[1])
        rel = self.rel(edge_type)

        return (src * rel * dst).sum(dim=1)

def train_step(batch, model, optimizer, num_neg_samples=5):
    model.train()
    optimizer.zero_grad()

    # Позитивные тройки
    h_pos, r_pos, t_pos = batch[:, 0], batch[:, 1], batch[:, 2]
    batch_size = h_pos.size(0)

    # Размножаем головы и отношения для негативных примеров
    h_neg = h_pos.repeat_interleave(num_neg_samples)
    r_neg = r_pos.repeat_interleave(num_neg_samples)

    # Генерируем случайные хвосты (по num_neg_samples для каждой позитивной тройки)
    num_nodes = model.emb.num_embeddings
    t_neg = torch.randint(0, num_nodes, (batch_size * num_neg_samples,), device=batch.device)

    # Объединяем позитивные и негативные примеры
    h = torch.cat([h_pos, h_neg])
    r = torch.cat([r_pos, r_neg])
    t = torch.cat([t_pos, t_neg])

    edge_label_index = torch.stack([h, t], dim=0)

    # Метки: 1 для позитивных, 0 для негативных
    labels = torch.cat([
        torch.ones(batch_size, device=batch.device),
        torch.zeros(batch_size * num_neg_samples, device=batch.device)
    ]).float()

    logits = model.decode(edge_label_index, r)
    loss_value = F.binary_cross_entropy_with_logits(logits, labels)

    loss_value.backward()
    optimizer.step()

    return loss_value.item()

import torch
from collections import defaultdict

def ranking_metrics_sampled_batch_kge(model, test_triplets, all_triplets, k_list=[1, 3, 10], num_negatives=1000, batch_size=256, N=10000):
    model.eval()
    num_nodes = model.emb.num_embeddings
    ranks = []
    test_triplets = test_triplets[:N]
    device = next(model.parameters()).device

    # 1. Строим словарь известных связей для фильтрации с учетом типа отношения (h, r)
    known_links = defaultdict(set)
    for i in range(all_triplets.size(0)):
        h, r, t = all_triplets[i].tolist()
        known_links[(h, r)].add(t)

    num_edges = test_triplets.size(0)

    with torch.no_grad():
        for start_idx in range(0, num_edges, batch_size):
            end_idx = min(start_idx + batch_size, num_edges)
            batch = test_triplets[start_idx:end_idx].to(device)

            h_batch = batch[:, 0]
            r_batch = batch[:, 1]
            t_batch = batch[:, 2]
            B = batch.size(0)

            # Генерация негативных хвостов (B, num_negatives)
            neg_tails = torch.randint(0, num_nodes, (B, num_negatives), device=batch.device)

            # Объединяем: на 0-й позиции истинный хвост, далее негативные -> (B, 1 + num_negatives)
            candidates = torch.cat([t_batch.unsqueeze(1), neg_tails], dim=1)

            # Получаем эмбеддинги и добавляем размерность для векторизованного умножения
            h_embs = model.emb(h_batch).unsqueeze(1) # (B, 1, D)
            r_embs = model.rel(r_batch).unsqueeze(1) # (B, 1, D)
            cand_embs = model.emb(candidates)        # (B, 1 + num_negatives, D)

            # Скоринг DistMult: (h * r * t).sum() по батчам и кандидатам
            scores = (h_embs * r_embs * cand_embs).sum(dim=2) # (B, 1 + num_negatives)

            # Фильтрация ложно-негативных примеров
            for b in range(B):
                h_idx = h_batch[b].item()
                r_idx = r_batch[b].item()
                known_set = known_links[(h_idx, r_idx)]

                cands = candidates[b].tolist()

                # Ищем индексы случайно сгенерированных хвостов, которые на самом деле верные
                # j != 0 гарантирует, что мы не занулим целевой истинный хвост
                mask_idx = [j for j, c in enumerate(cands) if c in known_set and j != 0]

                if mask_idx:
                    scores[b, mask_idx] = -float('inf')

            # Сортировка и поиск ранга истинного хвоста (он всегда на индексе 0)
            _, sorted_idx = torch.sort(scores, dim=1, descending=True)
            ranks_batch = (sorted_idx == 0).nonzero(as_tuple=True)[1] + 1
            ranks.extend(ranks_batch.tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = torch.mean(1.0 / ranks).item()

    hits = {}
    for k in k_list:
        hits[k] = torch.mean((ranks <= k).float()).item()

    return mrr, hits

In [39]:

df = pd.read_csv('../data/edges/clean_triples.csv')

data, node_map, pre2id = df_to_homo_two_rel(df)

all_positive_edges = data.edge_index

triplets = torch.stack([
    data.edge_index[0],
    data.edge_type,
    data.edge_index[1]
], dim=1)


num_triplets = triplets.size(0)
indices = torch.randperm(num_triplets)

train_size = int(0.8 * num_triplets)
val_size = int(0.1 * num_triplets)

test_indices = indices[train_size + val_size:]
val_indices = indices[train_size : train_size + val_size]
train_indices = indices[:train_size]

# Создаем сами выборки
train_triplets = triplets[train_indices]
val_triplets = triplets[val_indices]
test_triplets = triplets[test_indices]



train_loader = DataLoader(
    train_triplets,
    batch_size=4096,
    shuffle=True
)


In [41]:
model = DistMult(
    num_nodes = len(node_map),
    num_rels = len(pre2id),
    hidden_dim = 128,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for epoch in tqdm(range(1)):
    for batch in train_loader:
        batch = batch.to(device)

        loss_value = train_step(batch, model, optimizer, num_neg_samples=5)

    print(
        f"Epoch {epoch:03d} | "
        f"Loss {loss_value:.4f} | "
    )

100%|██████████| 1/1 [01:27<00:00, 87.20s/it]

Epoch 000 | Loss 0.7889 | 
